# CatBoost classifier 
with native categorical feature handling
(no one-hot encoding needed for Gender, City_Type, Current_Car_Type,
Home_Charging_Possible, Subsidy_Available, Range_Anxiety_Level).


In [2]:
import os
import sys
import importlib
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score

try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

sys.path.insert(0, BASE_DIR)

fe = importlib.import_module("06_Feature_Engineering")
cv = importlib.import_module("07_Cross_Validation")

MODEL_NAME = "catboost"
ARTIFACT_DIR = "./artifacts"
MODEL_DIR = "./models"
SUB_PATH = f"{ARTIFACT_DIR}/test_pred_{MODEL_NAME}.csv"
OOF_PATH = f"{ARTIFACT_DIR}/oof_{MODEL_NAME}.npy"

CATBOOST_PARAMS = dict(
    iterations=3000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=3.0,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=cv.SEED,
    verbose=200,
    early_stopping_rounds=150,
)

In [4]:
def main():
    os.makedirs(ARTIFACT_DIR, exist_ok=True)
    os.makedirs(MODEL_DIR, exist_ok=True)

    train, test = fe.load_raw_data()
    train = fe.engineer_features(train)
    test = fe.engineer_features(test)
    num_cols, cat_cols = fe.get_feature_lists(train)
    feature_cols = num_cols + cat_cols

    # CatBoost wants categorical columns as string (no NaNs here per EDA).
    for c in cat_cols:
        train[c] = train[c].astype(str)
        test[c] = test[c].astype(str)
    cat_feature_idx = [feature_cols.index(c) for c in cat_cols]

    y = train[fe.TARGET].values
    fold_ids = cv.get_or_create_folds(train, target_col=fe.TARGET)

    oof_pred = np.zeros(len(train))
    test_pred = np.zeros(len(test))
    fold_scores = []
    importances = np.zeros(len(feature_cols))

    print("=" * 70)
    print(f"CATBOOST ({cv.N_SPLITS}-fold CV)")
    print("=" * 70)

    for fold in range(cv.N_SPLITS):
        train_idx, valid_idx = cv.fold_split(train, fold_ids, fold)

        X_train, y_train = train.loc[train_idx, feature_cols], y[train_idx]
        X_valid, y_valid = train.loc[valid_idx, feature_cols], y[valid_idx]

        model = CatBoostClassifier(**CATBOOST_PARAMS)
        model.fit(
            X_train,
            y_train,
            cat_features=cat_feature_idx,
            eval_set=(X_valid, y_valid),
            use_best_model=True,
        )

        valid_pred = model.predict_proba(X_valid)[:, 1]
        oof_pred[valid_idx] = valid_pred

        fold_auc = roc_auc_score(y_valid, valid_pred)
        fold_scores.append(fold_auc)
        print(f"Fold {fold}: AUC = {fold_auc:.5f} (best_iter={model.get_best_iteration()})")

        test_pred += model.predict_proba(test[feature_cols])[:, 1] / cv.N_SPLITS
        importances += np.array(model.get_feature_importance()) / cv.N_SPLITS
        model.save_model(f"{MODEL_DIR}/catboost_fold{fold}.cbm")

    print(f"\nMean fold AUC: {np.mean(fold_scores):.5f} (+/- {np.std(fold_scores):.5f})")
    cv.summarize_oof(y, oof_pred, MODEL_NAME)

    imp_df = pd.DataFrame({"feature": feature_cols, "importance": importances})
    imp_df = imp_df.sort_values("importance", ascending=False)
    print("\nTop feature importances:")
    print(imp_df.head(15).to_string(index=False))

    np.save(OOF_PATH, oof_pred)
    pd.DataFrame({fe.ID_COL: test[fe.ID_COL], fe.TARGET: test_pred}).to_csv(
        SUB_PATH, index=False
    )
    print(f"\nSaved OOF predictions -> {OOF_PATH}")
    print(f"Saved test predictions -> {SUB_PATH}")


In [5]:
if __name__ == "__main__":
    main()

CATBOOST (5-fold CV)
0:	test: 0.9305668	best: 0.9305668 (0)	total: 241ms	remaining: 12m 3s
200:	test: 0.9391651	best: 0.9391651 (200)	total: 42.3s	remaining: 9m 49s
400:	test: 0.9398447	best: 0.9398447 (400)	total: 1m 25s	remaining: 9m 12s
600:	test: 0.9400797	best: 0.9400797 (600)	total: 2m 7s	remaining: 8m 27s
800:	test: 0.9401851	best: 0.9401918 (791)	total: 2m 49s	remaining: 7m 44s
1000:	test: 0.9401782	best: 0.9402203 (887)	total: 3m 31s	remaining: 7m 2s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.9402202561
bestIteration = 887

Shrink model to first 888 iterations.
Fold 0: AUC = 0.94022 (best_iter=887)
0:	test: 0.9314825	best: 0.9314825 (0)	total: 218ms	remaining: 10m 52s
200:	test: 0.9401959	best: 0.9401959 (200)	total: 43.9s	remaining: 10m 11s
400:	test: 0.9408396	best: 0.9408396 (400)	total: 1m 25s	remaining: 9m 17s
600:	test: 0.9410522	best: 0.9410558 (599)	total: 2m 8s	remaining: 8m 33s
800:	test: 0.9410629	best: 0.9410717 (687)	total: 2m 50s	remaini